# TradingAgents adım adım

Bu defter tek bir analizi çalıştırır ve her ajanın ne ürettiğini sırayla gösterir. Ayrıntılı anlatım için `docs/tr/NASIL_CALISIYOR.md` dosyasına bak.

**Başlamadan önce** terminalde bir kez `tradingagents login chatgpt` çalıştırıp ChatGPT hesabınla giriş yap.

## 1. Ayarlar

Planının sunduğu modelleri listeleyip aşağıdaki iki model kimliğini oradan seç.

In [ ]:
from tradingagents.llm_clients import chatgpt_auth

for name, slug in chatgpt_auth.list_models():
    print(f"{slug:30} {name}")

In [ ]:
from tradingagents.default_config import DEFAULT_CONFIG

config = DEFAULT_CONFIG.copy()
config["llm_provider"] = "chatgpt"
config["quick_think_llm"] = "BURAYA_HIZLI_MODEL"   # analistler, araştırmacılar, trader
config["deep_think_llm"] = "BURAYA_GUCLU_MODEL"    # Research Manager ve Portfolio Manager
config["output_language"] = "Turkish"
config["max_debate_rounds"] = 1        # boğa/ayı tartışma turu
config["max_risk_discuss_rounds"] = 1  # risk ekibi tartışma turu

TICKER = "NVDA"          # kripto için "BTC-USD", altın için "GC=F"
ASSET_TYPE = "stock"     # kripto için "crypto"
DATE = "2026-10-20"      # analiz tarihi; gelecek bir tarih olamaz
ANALYSTS = ["market", "news", "fundamentals"]  # duygu analisti: "social"; emtia için ["market", "news"]

## 2. Grafiği kur ve çalıştır

Bu hücre birkaç dakika sürebilir: her ajan sırayla LLM'e soru soruyor.

In [ ]:
from tradingagents.graph.trading_graph import TradingAgentsGraph

ta = TradingAgentsGraph(selected_analysts=ANALYSTS, config=config)
state, rating = ta.propagate(TICKER, DATE, asset_type=ASSET_TYPE)
print("Son not:", rating)

## 3. Analist raporları

Analistler aynı anda çalıştı. Her biri kendi araçlarıyla veri çekip bir rapor yazdı.

In [ ]:
from IPython.display import Markdown, display

for key, title in [("market_report", "Market (teknik)"), ("sentiment_report", "Sentiment"),
                   ("news_report", "Haberler"), ("fundamentals_report", "Temel analiz")]:
    if state.get(key):
        display(Markdown(f"### {title}\n\n{state[key]}"))

## 4. Boğa ve ayı tartışması

İki araştırmacı analist raporlarını kullanarak lehte ve aleyhte tartıştı.

In [ ]:
debate = state["investment_debate_state"]
display(Markdown("### Boğa\n\n" + debate["bull_history"]))
display(Markdown("### Ayı\n\n" + debate["bear_history"]))

## 5. Research Manager planı ve Trader önerisi

In [ ]:
display(Markdown("### Yatırım planı\n\n" + state["investment_plan"]))
display(Markdown("### Trader önerisi\n\n" + state["trader_investment_plan"]))

## 6. Risk ekibi

Aynı öneriyi agresif, muhafazakâr ve nötr gözle eleştirdiler.

In [ ]:
risk = state["risk_debate_state"]
for key, title in [("aggressive_history", "Agresif"), ("conservative_history", "Muhafazakâr"),
                   ("neutral_history", "Nötr")]:
    display(Markdown(f"### {title}\n\n{risk[key]}"))

## 7. Portfolio Manager'ın son kararı

In [ ]:
display(Markdown(state["final_trade_decision"]))
print("Not:", state["final_rating"])

## 8. Raporu kaydet

`complete_report.html` dosyasını tarayıcıda açarak hepsini tek sayfada okuyabilirsin.

In [ ]:
path = ta.save_reports(state, TICKER)
print(path)

## Şimdi kurcala

- `max_debate_rounds` değerini 3 yap ve tekrar çalıştır. Karar değişti mi?
- `ANALYSTS` listesinden bir analisti çıkar. Hangisi kararı en çok etkiliyor?
- Aynı hücreyi iki kez çalıştır. LLM'ler her seferinde aynı cevabı vermez; sonuç ne kadar tutarlı?